# DDP on one GPU — a working simulation

**Setup:** 1 GPU, 8GB VRAM, Python 3.12, CUDA 13.0.

This notebook simulates `DistributedDataParallel` (DDP) using **2 processes that both target your one GPU**. The processes coordinate over the `gloo` backend (CPU-side collectives) while the model and tensors stay on `cuda:0`. That split — `gloo` for coordination, `cuda:0` for compute — is the standard way to exercise the real DDP API without a second GPU.

Everything below is the same API you'd use with `nccl` across real GPUs; only the backend and `MASTER_*` env vars change when you move to a cluster.

References:
- PyTorch, *Getting Started with Distributed Data Parallel* — https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html
- Li et al., *PyTorch Distributed: Experiences on Accelerating Data Parallel Training*, VLDB 2020 — https://arxiv.org/abs/2006.15704


## 0. Environment check

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 1. Setup / cleanup for the process group

Each spawned process calls `setup(rank, world_size)` before doing anything else, and `cleanup()` when done. `MASTER_ADDR`/`MASTER_PORT` are how the processes find each other — on a real cluster these point at node 0; here they're both `localhost` since everything runs on your machine.

We use `backend="gloo"` deliberately. `nccl` assumes one rank per physical GPU and doesn't reliably support two ranks sharing one device — `gloo` has no such restriction, so it's the right tool for this single-GPU simulation.

In [ ]:
import os
import torch.distributed as dist

def setup(rank: int, world_size: int):
    os.environ.setdefault("MASTER_ADDR", "localhost")
    os.environ.setdefault("MASTER_PORT", "29500")
    dist.init_process_group(backend="gloo", rank=rank, world_size=world_size)
    torch.cuda.set_device(0)  # both ranks share the one physical GPU

def cleanup():
    dist.destroy_process_group()


## 2. A small model and a synthetic dataset

Kept deliberately tiny (a 3-layer CNN, random tensors standing in for images) so this trains in seconds and never comes close to 8GB — the point of this stage is the DDP mechanics, not model capacity.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 28 -> 14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 14 -> 7
        x = x.flatten(1)
        return self.fc(x)

class SyntheticImages(Dataset):
    """Stand-in for a real dataset: random 28x28 'images' + random labels.
    Fixed seed so every rank builds the identical underlying dataset;
    the DistributedSampler is what splits it across ranks."""
    def __init__(self, n=2000, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.randn(n, 1, 28, 28, generator=g)
        self.y = torch.randint(0, 10, (n,), generator=g)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


## 3. The training function each rank runs

This is the part that matters. Compare it mentally to a normal single-GPU loop — the additions are exactly four things:

1. `setup(rank, world_size)` at the top
2. `DistributedSampler` instead of `shuffle=True`
3. `model = DDP(model.to(device), device_ids=[0])`
4. `sampler.set_epoch(epoch)` each epoch, so the shuffle differs per epoch

Nothing else about the loop changes — DDP's autograd hooks handle the all-reduce automatically inside `loss.backward()`.

In [ ]:
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.optim as optim

def train(rank: int, world_size: int, epochs: int = 2):
    setup(rank, world_size)
    device = torch.device("cuda:0")

    dataset = SyntheticImages(n=2000, seed=0)          # identical on every rank
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=0)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler)

    torch.manual_seed(0)                                # identical init on every rank
    model = TinyCNN().to(device)
    model = DDP(model, device_ids=[0])
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        sampler.set_epoch(epoch)                        # different shuffle each epoch, still deterministic
        running_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()                             # gradient all-reduce happens inside here
            optimizer.step()
            running_loss += loss.item()
        print(f"[rank {rank}] epoch {epoch} avg loss {running_loss / len(loader):.4f}")

    # Proof of sync: every rank trained on a DIFFERENT data shard, but should have
    # BIT-IDENTICAL weights after the all-reduce'd update.
    checksum = sum(p.detach().sum().item() for p in model.parameters())
    print(f"[rank {rank}] parameter checksum after training: {checksum:.6f}")

    cleanup()


## 4. Spawn the two ranks

`mp.spawn` launches `world_size` processes, each running `train(rank, world_size)`. This is the single-machine equivalent of what `torchrun` does across multiple machines — you'll swap this exact call for `torchrun --nproc_per_node=N script.py` once you have real multi-GPU hardware.

**What to look for in the output:** both ranks print a different average loss per epoch (they trained on different data), but the two parameter checksums at the end should match almost exactly — that's the all-reduce doing its job.

In [ ]:
import torch.multiprocessing as mp

def run_ddp_demo(world_size: int = 2, epochs: int = 2):
    mp.spawn(train, args=(world_size, epochs), nprocs=world_size, join=True)

if __name__ == "__main__":
    run_ddp_demo(world_size=2, epochs=2)


> **Notebook note:** `mp.spawn` sometimes behaves inconsistently inside Jupyter on certain platforms (pickling of interactively-defined functions). If the cell above hangs or errors, save cells 0–4 as `ddp_demo.py` and run `python ddp_demo.py` from a terminal instead — the code is unchanged, only the launch context differs.

## 5. Scaling past today's setup

You won't run this part — it's the mental checklist for the day you get real multi-GPU hardware.

| Today (1 GPU, simulated) | Real multi-GPU / multi-node |
|---|---|
| `backend="gloo"` | `backend="nccl"` |
| both ranks on `cuda:0` | one rank per physical GPU (`device_ids=[local_rank]`) |
| launched with `mp.spawn` | launched with `torchrun --nproc_per_node=N` |
| `MASTER_ADDR="localhost"` | `MASTER_ADDR` = node 0's actual address |

**If 8GB is the ceiling even with one real GPU per rank:**
- Mixed precision — wrap the forward/backward in `torch.autocast("cuda")` with a `torch.cuda.amp.GradScaler`; roughly halves activation memory.
- Gradient accumulation — take several micro-batches before `optimizer.step()`. Wrap the micro-steps (all but the last) in `with model.no_sync():` so DDP skips the all-reduce until the real step; skipping this silently disables the memory savings by syncing every micro-batch anyway.
- If the *model itself* — not just the batch — doesn't fit in 8GB, that's a job for **FSDP** (Fully Sharded Data Parallel), not DDP: https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html


## 6. Self-check

Answer without scrolling up:

1. What exactly does DDP synchronize between ranks?
2. Why does `DistributedSampler` matter here, given both ranks build the *same* `SyntheticImages` dataset?
3. What would happen to memory savings if you forgot `model.no_sync()` during gradient accumulation?
4. What two things change when you move this code from 1 simulated GPU to 4 real GPUs?
